# Oscilaciones inerciales

**Alejandro Jaramillo Moreno**  
Instituto de Ciencias de la Atmósfera y Cambio Climático  
Universidad Nacional Autónoma de México

Las oscilaciones inerciales corresponden al movimiento de una partícula en la atmósfera bajo la influencia exclusiva de la fuerza de Coriolis, en ausencia de fuerzas de presión y fricción.

Este tipo de movimiento surge al considerar las ecuaciones horizontales simplificadas:

$$
\frac{du}{dt} = f v
$$

$$
\frac{dv}{dt} = -f u
$$

donde:

$$
f = 2\Omega \sin\phi
$$

es el parámetro de Coriolis.

El sistema completo incluye también la evolución de la posición:

$$
\frac{dx}{dt} = u, \qquad \frac{dy}{dt} = v
$$

Por lo tanto, se tiene un sistema acoplado para:

- posición: $(x, y)$  
- velocidad: $(u, v)$  

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from IPython.display import HTML

# Velocidad angular de rotación de la Tierra [rad/s]
Omega = 7.292e-5               
# radio de la Tierra [m]
Re = 6371e3               

In [2]:
# latitud inicial [grados]
lat0_deg = 19.0          
# longitud inicial [grados]
lon0_deg = -99    

# velocidad inicial [m/s]
V = 10.0           


# número de pasos de integración
# este valor es para hacer la animación mas rápida
# pero al costo de un menor número de pasos
n_pasos = 100

In [3]:
lat0 = np.deg2rad(lat0_deg)
lon0 = np.deg2rad(lon0_deg)

# parámetro de Coriolis en la latitud inicial [1/s]
f0 = 2*Omega*np.sin(lat0)   

# posición inicial en el plano local [m]
x0, y0 = 0.0, 0.0    

# Periodo inercial basado en f0
# Mostramos en clase que este movimiento produce una oscilación con
# periodo t=2Pi/f. Usaremos esto para calcular una oscilación completa 
# para construir una animación
T = 2*np.pi/abs(f0)   # [s]

# Integramos un periodo completo
t_eval = np.linspace(0, T, n_pasos)

# Radio teórico
R = V/abs(f0)

print(f"Latitud inicial = {lat0_deg:.2f}°")
print(f"Longitud inicial = {lon0_deg:.2f}°")
print(f"f0 = {f0:.6e} s^-1")
print(f"Periodo inercial = {T/3600:.2f} h")
print(f"Radio inercial = {R/1000:.2f} km")

Latitud inicial = 19.00°
Longitud inicial = -99.00°
f0 = 4.748086e-05 s^-1
Periodo inercial = 36.76 h
Radio inercial = 210.61 km


In [4]:
def inertial_system_fconst(t, Y):
    x, y, u, v = Y

    dxdt = u
    dydt = v
    dudt = f0*v
    dvdt = -f0*u

    return [dxdt, dydt, dudt, dvdt]

def inertial_system_fvar(t, Y):
    
    x, y, u, v = Y
    lat_t = lat0+y/Re
    f_i = 2*Omega*np.sin(lat_t)
    dxdt = u
    dydt = v
    dudt = f_i*v
    dvdt = -f_i*u
    
    return [dxdt, dydt, dudt, dvdt]

In [5]:
# Condiciones iniciales
Y0 = [x0, y0, V, 0.0]

# Solución numérica
sol_cte = solve_ivp(inertial_system_fconst, [0, T], Y0, t_eval=t_eval)
print(sol_cte)

sol_var = solve_ivp(inertial_system_fvar, [0, T], Y0, t_eval=t_eval)
print(sol_var)

  message: The solver successfully reached the end of the integration interval.
  success: True
   status: 0
        t: [ 0.000e+00  1.337e+03 ...  1.310e+05  1.323e+05]
        y: [[ 0.000e+00  1.336e+04 ... -1.299e+04  3.678e+02]
            [ 0.000e+00 -4.240e+02 ... -4.094e+02 -1.117e+01]
            [ 1.000e+01  9.980e+00 ...  9.981e+00  9.999e+00]
            [ 0.000e+00 -6.342e-01 ...  6.170e-01 -1.746e-02]]
      sol: None
 t_events: None
 y_events: None
     nfev: 92
     njev: 0
      nlu: 0
  message: The solver successfully reached the end of the integration interval.
  success: True
   status: 0
        t: [ 0.000e+00  1.337e+03 ...  1.310e+05  1.323e+05]
        y: [[ 0.000e+00  1.336e+04 ... -2.498e+05 -2.406e+05]
            [ 0.000e+00 -4.240e+02 ... -7.200e+04 -6.230e+04]
            [ 1.000e+01  9.980e+00 ...  6.636e+00  7.083e+00]
            [ 0.000e+00 -6.342e-01 ...  7.467e+00  7.045e+00]]
      sol: None
 t_events: None
 y_events: None
     nfev: 98
     njev: 0

In [6]:
def xy_to_latlon(x_arr, y_arr):
    lat = lat0+y_arr/Re
    lon = lon0+x_arr/(Re*np.cos(lat0))
    return np.rad2deg(lat), np.rad2deg(lon)


x_cte = sol_cte.y[0]
y_cte = sol_cte.y[1]   
u_cte = sol_cte.y[2]
v_cte = sol_cte.y[3]
t_cte = sol_cte.t

x_var = sol_var.y[0]
y_var = sol_var.y[1]   
u_var = sol_var.y[2]
v_var = sol_var.y[3]
t_var = sol_var.t


lat_cte, lon_cte = xy_to_latlon(x_cte, y_cte)
lat_var, lon_var = xy_to_latlon(x_var, y_var)

# Centro teórico del círculo en coordenadas locales
yc_center = y0-V/f0
xc_center = x0

# Convertir también el centro a lat/lon
lat_center = lat0+yc_center/Re
lon_center = lon0+xc_center/(Re*np.cos(lat0))

lat_center_deg = np.rad2deg(lat_center)
lon_center_deg = np.rad2deg(lon_center)

In [7]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title("Oscilación inercial (solución numérica)")
ax.grid()

all_lon = np.concatenate([lon_cte, lon_var])
all_lat = np.concatenate([lat_cte, lat_var])
lon_margin = 0.1 * (all_lon.max() - all_lon.min())
lat_margin = 0.1 * (all_lat.max() - all_lat.min())
if lon_margin == 0:
    lon_margin = 0.01
if lat_margin == 0:
    lat_margin = 0.01
ax.set_xlim(all_lon.min() - lon_margin, all_lon.max() + lon_margin)
ax.set_ylim(all_lat.min() - lat_margin, all_lat.max() + lat_margin)

# Trayectorias completas tenues
ax.plot(lon_cte, lat_cte, '--', alpha=0.3, label="Trayectoria f cte")
ax.plot(lon_var, lat_var, '--', color='firebrick', alpha=0.3, label="Trayectoria f variable")
# Punto inicial y centro teórico
ax.plot(lon0_deg, lat0_deg, 'go', ms=6, label="Punto inicial")
ax.plot(lon_center_deg, lat_center_deg, 'ko', ms=4, label="Centro teórico")

# Elementos animados — f constante (azul, por defecto)
line,      = ax.plot([], [], lw=2, label="Recorrido f cte")
point,     = ax.plot([], [], 'ro', ms=8)
# Elementos animados — f variable (rojo oscuro)
line_var,  = ax.plot([], [], lw=2, color='firebrick', label="Recorrido f variable")
point_var, = ax.plot([], [], 'rs', ms=8)

time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
#ax.legend(loc="lower right")

def init():
    line.set_data([], [])
    point.set_data([], [])
    line_var.set_data([], [])
    point_var.set_data([], [])
    time_text.set_text("")
    return line, point, line_var, point_var, time_text

def update(frame):
    line.set_data(lon_cte[:frame+1], lat_cte[:frame+1])
    point.set_data([lon_cte[frame]], [lat_cte[frame]])
    line_var.set_data(lon_var[:frame+1], lat_var[:frame+1])
    point_var.set_data([lon_var[frame]], [lat_var[frame]])
    time_text.set_text(f"t = {t_cte[frame]/3600:.2f} h")
    return line, point, line_var, point_var, time_text

anim = FuncAnimation(
    fig,
    update,
    frames=len(t_cte),
    init_func=init,
    interval=50,
    blit=True
)
plt.close(fig)
HTML(anim.to_jshtml())